# Wizualizacja wybranego modelu

Wczytuje zapisany model (wyeksportowany z `report.ipynb` lub `main.py`) i udostępnia interaktywne wizualizacje:
- Architektura sieci (3D)
- Krzywe uczenia (jeśli dostępne)
- Interaktywny podgląd predykcji na zbiorze testowym


In [11]:
# ── Setup ─────────────────────────────────────────────────────────────
from pathlib import Path
import sys
import numpy as np

ROOT_DIR = Path('').resolve().parents[0]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from Code.train.model import Model
from Code.Import_training_data.import_traing_data import DataLoader

# Load test data for predictions
X_train, y_train = DataLoader.get_train_data()
X_val,   y_val   = DataLoader.get_val_data()
X_test,  y_test  = DataLoader.get_test_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')


Train: (6285, 64)  Val: (270, 64)  Test: (270, 64)


In [12]:
# ── Load saved model ──────────────────────────────────────────────────
# Change this path to load a different saved model
MODEL_PATH = 'saved_models/bayes_best_model'

model = Model.load(MODEL_PATH)

# Derive architecture info from the loaded model
layer_sizes = [model.layers[0].num_inputs] + [l.num_nodes for l in model.layers]
activations = [l.activation_name for l in model.layers]

val_acc  = model.accuracy(X_val,  y_val)
test_acc = model.accuracy(X_test, y_test)

print(f"Loaded : {MODEL_PATH}.npz")
print(f"Architecture : {layer_sizes}")
print(f"Activations  : {activations}")
print(f"Val accuracy : {val_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


Loaded : saved_models/bayes_best_model.npz
Architecture : [64, 179, 10]
Activations  : ['relu', 'softmax']
Val accuracy : 0.9778
Test accuracy: 0.9667


---
## Architektura sieci (3D)


In [13]:
from Code.Visualization.network_visualization import show_architecture_3d

show_architecture_3d(layer_sizes, activations)


---
## Macierz pomyłek


In [14]:
import plotly.graph_objects as go

y_pred = np.argmax(model._forward(X_test), axis=1)
n_classes = 10
cm = np.zeros((n_classes, n_classes), dtype=int)
for true, pred in zip(y_test, y_pred):
    cm[true][pred] += 1

fig_cm = go.Figure(go.Heatmap(
    z=cm,
    x=[str(i) for i in range(n_classes)],
    y=[str(i) for i in range(n_classes)],
    colorscale='Blues',
    text=cm, texttemplate='%{text}',
    hovertemplate='True: %{y}<br>Pred: %{x}<br>Count: %{z}<extra></extra>',
))
fig_cm.update_layout(
    title=f'Confusion Matrix  (test acc={test_acc:.4f})',
    xaxis_title='Predicted', yaxis_title='True',
    plot_bgcolor='#1e1e2e', paper_bgcolor='#1e1e2e',
    font=dict(color='white'), height=500,
    yaxis=dict(autorange='reversed'),
)
fig_cm.show()


---
## Interaktywny podgląd predykcji

Wybierz cyfrę z dropdown i kliknij **▶ Losuj próbkę**, aby zobaczyć przykład z zestawu testowego i predykcję modelu.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots

_digit_dd = widgets.Dropdown(
    options=[('All', -1)] + [(str(i), i) for i in range(10)],
    value=-1,
    description='Cyfra:',
    layout=widgets.Layout(width='130px'),
)
_sample_btn = widgets.Button(
    description='▶ Losuj próbkę',
    button_style='primary',
    layout=widgets.Layout(width='150px'),
)
_out = widgets.Output()

def _on_sample(_):
    digit = _digit_dd.value
    if digit == -1:
        idx = np.random.randint(len(X_test))
    else:
        indices = np.where(y_test == digit)[0]
        idx = np.random.choice(indices)

    x    = X_test[idx].reshape(1, -1)
    pr   = model._forward(x)[0]
    p    = int(np.argmax(pr))
    true = int(y_test[idx])
    img  = X_test[idx].reshape(8, 8)

    correct = p == true
    verdict = '✓ Poprawnie' if correct else f'✗ Błąd (prawdziwa: {true})'

    with _out:
        _out.clear_output(wait=True)
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.35, 0.65],
            subplot_titles=(f'Wejście (true={true})', f'Predykcja: {p}  {verdict}'),
        )
        fig.add_trace(go.Heatmap(
            z=img[::-1], colorscale='gray', showscale=False,
            hovertemplate='Wiersz %{y}, Kolumna %{x}<br>Wartość: %{z}<extra></extra>',
        ), row=1, col=1)
        fig.add_trace(go.Bar(
            x=list(range(10)), y=pr.tolist(),
            text=[f'{v:.0%}' for v in pr], textposition='outside',
            marker_color=['#00CC96' if i == p else '#636EFA' for i in range(10)],
            hovertemplate='Cyfra %{x}: %{y:.4f}<extra></extra>',
        ), row=1, col=2)
        fig.update_layout(
            height=320,
            plot_bgcolor='#1e1e2e', paper_bgcolor='#1e1e2e',
            font=dict(color='white'), margin=dict(t=50, b=30), showlegend=False,
        )
        fig.update_xaxes(gridcolor='#333', row=1, col=2, tickmode='array', tickvals=list(range(10)))
        fig.update_yaxes(gridcolor='#333', range=[0, 1.2], row=1, col=2)
        fig.update_xaxes(showticklabels=False, row=1, col=1)
        fig.update_yaxes(showticklabels=False, row=1, col=1)
        fig.show()

_sample_btn.on_click(_on_sample)
display(widgets.HBox([_digit_dd, _sample_btn]))
display(_out)
_on_sample(None)


Output()